In [ ]:
import imageio.v3 as iio
from pathlib import Path
from PIL import Image, ImageDraw, ImageSequence, ImageFont
import numpy as np
import re

In [47]:
def add_timestamp(image_array, label):
    img = Image.fromarray(image_array)
    draw = ImageDraw.Draw(img)
    
    try:
        font = ImageFont.truetype("arial.ttf", 48)
    except:
        font = ImageFont.load_default()
    
    # Draw black outline for readability
    x, y = 10, 10
    for dx, dy in [(-1,-1),(1,-1),(-1,1),(1,1)]:
        draw.text((x+dx, y+dy), label, font=font, fill=(0, 0, 0))
    
    # Draw red text on top
    draw.text((x, y), label, font=font, fill=(255, 0, 0))
    
    return np.array(img)

In [48]:
def add_red_box(input_path, output_path, box_coords, line_width=3):
    """
    box_coords: (x1, y1, x2, y2) - top-left and bottom-right corners
    """
    img = Image.open(input_path)
    frames = []

    for frame in ImageSequence.Iterator(img):
        frame = frame.convert("RGBA")
        draw = ImageDraw.Draw(frame)
        draw.rectangle(box_coords, outline="red", width=line_width)
        frames.append(frame)

    frames[0].save(
        output_path,
        save_all=True,
        append_images=frames[1:],
        loop=img.info.get("loop", 0)
    )

In [49]:
# Get all jpg files
files = sorted(Path('../assets/cabrini_green_google_earth/').glob('*.jpg'))  # sorted for consistent order

In [50]:
frames = []
for f in files:
    # Extract YYYY_mm from filename ending
    match = re.search(r'_(\d{4})_(\d{2})$', f.stem)
    if match:
        year, month = match.groups()
        label = f"{year}-{month}"
    else:
        label = f.stem  # fallback to full filename if no match
    
    image = iio.imread(f)
    stamped = add_timestamp(image, label)
    frames.append(stamped)
    
iio.imwrite('../assets/cabrini_green_google_earth/cabrini_green_2000_2020.gif', frames, duration=1000, loop=0)


In [63]:
# Add red boxes

# Box around Target
add_red_box(
    '../assets/cabrini_green_google_earth/cabrini_green_2000_2020.gif', 
    '../assets/cabrini_green_google_earth/cabrini_green_2000_2020_annotated.gif', 
    box_coords=(100, 175, 450, 350)
)

# Box around Evergreen
add_red_box(
    '../assets/cabrini_green_google_earth/cabrini_green_2000_2020_annotated.gif', 
    '../assets/cabrini_green_google_earth/cabrini_green_2000_2020_annotated.gif', 
    box_coords=(455, 200, 650, 340)
)

# Box around Parkside of Old Town
add_red_box(
    '../assets/cabrini_green_google_earth/cabrini_green_2000_2020_annotated.gif', 
    '../assets/cabrini_green_google_earth/cabrini_green_2000_2020_annotated.gif', 
    box_coords=(455, 345, 665, 475)
)